# Import packages

In [1]:
import os
import subprocess
from google.colab import files

# Install tools

In [2]:
!apt-get -y install samtools
!pip -q install deeptools

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libhts3 libhtscodecs2
Suggested packages:
  cwltool
The following NEW packages will be installed:
  libhts3 libhtscodecs2 samtools
0 upgraded, 3 newly installed, 0 to remove and 6 not upgraded.
Need to get 963 kB of archives.
After this operation, 2,270 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libhtscodecs2 amd64 1.1.1-3 [53.2 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libhts3 amd64 1.13+ds-2build1 [390 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 samtools amd64 1.13-4 [520 kB]
Fetched 963 kB in 2s (470 kB/s)
Selecting previously unselected package libhtscodecs2:amd64.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../libhtscodecs2_1.1.1-3_amd64.deb ...
Unpacking libhtscodecs2:amd64 (1.1.1-3

In [3]:
!deeptools --version
!samtools --version

deeptools 3.5.6
samtools 1.13
Using htslib 1.13+ds
Copyright (C) 2021 Genome Research Ltd.

Samtools compilation details:
    Features:       build=configure curses=yes 
    CC:             gcc
    CPPFLAGS:       -frelease  -Wdate-time -D_FORTIFY_SOURCE=2
    CFLAGS:         -g -O2 -ffile-prefix-map=�BUILDPATH�=. -flto=auto -ffat-lto-objects -fstack-protector-strong -Wformat -Werror=format-security
    LDFLAGS:        -Wl,-Bsymbolic-functions -flto=auto -Wl,-z,relro -Wl,-z,now
    HTSDIR:         
    LIBS:           
    CURSES_LIB:     -lcurses

HTSlib compilation details:
    Features:       build=configure plugins=yes, plugin-path=/usr/local/lib/htslib:/usr/local/libexec/htslib:: libcurl=yes S3=yes GCS=yes libdeflate=yes lzma=yes bzip2=yes htscodecs=1.1.1
    CC:             gcc
    CPPFLAGS:       -I. -DSAMTOOLS=1 -Wdate-time -D_FORTIFY_SOURCE=2
    CFLAGS:         -g -O2 -ffile-prefix-map=/build/htslib-TQtOKr/htslib-1.13+ds=. -flto=auto -ffat-lto-objects -fstack-protector-strong

# Download bam files

In [4]:
input_dir = 'input/chip_seq'
os.makedirs(input_dir, exist_ok=True)

In [5]:
# Replace URLs with actual ENCODE links
files_dict = {
    'replicate_1.bam': 'https://www.encodeproject.org/files/ENCFF595MHF/@@download/ENCFF595MHF.bam',
    'replicate_2.bam': 'https://www.encodeproject.org/files/ENCFF084FDL/@@download/ENCFF084FDL.bam',
    'control_1.bam': 'https://www.encodeproject.org/files/ENCFF959XDT/@@download/ENCFF959XDT.bam',
    'control_2.bam': 'https://www.encodeproject.org/files/ENCFF016WME/@@download/ENCFF016WME.bam',
}

for filename, url in files_dict.items():
    out_path = os.path.join(input_dir, filename)

    if os.path.exists(out_path):
        print(f'Skipping {out_path}')
        continue

    subprocess.run(['wget', '-O', out_path, url], check=True)

# Index bam

In [6]:
for filename in os.listdir(input_dir):
    if filename.endswith(".bam"):
        bam_path = os.path.join(input_dir, filename)
        print(f"Indexing {bam_path}...")
        subprocess.run(["samtools", "index", bam_path], check=True)

Indexing input/chip_seq/control_2.bam...
Indexing input/chip_seq/control_1.bam...
Indexing input/chip_seq/replicate_2.bam...
Indexing input/chip_seq/replicate_1.bam...


# Convert bam into bigWig
Codes below took 32 min to finish

In [7]:
output_dir = 'output/chip_seq'
os.makedirs(output_dir, exist_ok=True)

In [8]:
for filename in os.listdir(input_dir):
    if filename.endswith(".bam"):
        bam_path = os.path.join(input_dir, filename)

        # output name: chip.bam → chip.bw
        bw_name = filename.replace(".bam", ".bw")
        bw_path = os.path.join(output_dir, bw_name)

        print(f"Processing {filename}...")

        subprocess.run([
            "bamCoverage",
            "-b", bam_path,
            "-o", bw_path,
            "--normalizeUsing", "CPM",
            "--binSize", "25",
            "-p", "max"
        ], check=True)

# Now bigWig files can be import into UCSC

Processing control_2.bam...
Processing control_1.bam...
Processing replicate_2.bam...
Processing replicate_1.bam...


# Download bigWig

In [10]:
!zip -r chip_seq_bigWig.zip output/chip_seq

files.download('chip_seq_bigWig.zip')

  adding: output/chip_seq/ (stored 0%)
  adding: output/chip_seq/replicate_1.bw (deflated 4%)
  adding: output/chip_seq/control_2.bw (deflated 4%)
  adding: output/chip_seq/control_1.bw (deflated 4%)
  adding: output/chip_seq/replicate_2.bw (deflated 3%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>